# Imports

In [1]:
import pandas as pd
import numpy as np
from cyvcf2 import VCF

# Clumping

In [2]:
def clump(input_file_path, output_file_path, p1=5e-8, p2=0.01, kb_window=250, r2_threshold=0.5):
    # Load results
    gwas = pd.read_csv(input_file_path, sep=r"\s+")
    gwas_filtered = gwas[gwas["P"] <= p2].copy()

    # SNPs actually needed for LD
    needed_snps = set(gwas_filtered["SNP"])

    # Load VCF and keep only needed SNPs
    vcf = VCF("../data/ps3_gwas_deduped.vcf.gz")
    geno_dict = {}
    seen_variants = set()
    for v in vcf:
        # Skip if not in GWAS results
        if v.ID not in needed_snps:
            continue
        
        # Check for and skip duplicates
        variant_key = (v.CHROM, v.POS, v.REF, ",".join(v.ALT))
        if variant_key in seen_variants:
            continue
        seen_variants.add(variant_key)

        # Convert to dosage
        g = v.gt_types.astype(float)
        missing_mask = (g == 2)  # UNKNOWN
        g[g == 3] = 2  # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan  # UNKNOWN -> NaN

        geno_dict[v.ID] = g

    # Define function to find LD for a pair
    def ld_r2(snp1, snp2):
        g1 = geno_dict[snp1]
        g2 = geno_dict[snp2]

        # Filter out NaNs and check if at least 3 overlapping samples
        mask = ~np.isnan(g1) & ~np.isnan(g2)
        if mask.sum() < 3:
            return np.nan

        # Correlation coefficient calculation
        x = g1[mask]
        y = g2[mask]

        x_mean = x.mean()
        y_mean = y.mean()

        cov = np.sum((x - x_mean) * (y - y_mean))
        var_x = np.sum((x - x_mean) ** 2)
        var_y = np.sum((y - y_mean) ** 2)

        if var_x == 0 or var_y == 0:
            return np.nan

        r = cov / np.sqrt(var_x * var_y)

        return r * r

    lead_snps = []
    lead_to_sp2 = {}
    removed = set()
    bp_window = kb_window * 1000

    # Process each chromosome separately
    for chrom, df_chr in gwas_filtered.groupby("CHR"):
        # Sort by BP for left/right scanning within chromosome
        df_chr_bp = df_chr.sort_values("BP").reset_index(drop=True)

        # Choose lead SNPs in p-value order within chromosome
        order = df_chr_bp.sort_values("P").index

        for idx in order:
            row = df_chr_bp.loc[idx]
            row_snp = row["SNP"]

            # Check if already clumped
            if row_snp in removed:
                continue

            # Only SNPs passing p1 can start a clump
            if row["P"] > p1:
                continue

            # Add as lead SNP
            lead_snps.append(row_snp)
            lead_to_sp2[row_snp] = []
            row_bp = row["BP"]

            # Scan left side of window
            j = idx - 1
            while j >= 0:
                other = df_chr_bp.loc[j]
                other_snp = other["SNP"]

                dist_bp = row_bp - other["BP"]
                if dist_bp > bp_window:
                    break

                if other_snp not in removed:
                    r2 = ld_r2(row_snp, other_snp)
                    if not np.isnan(r2) and r2 >= r2_threshold:
                        removed.add(other_snp)
                        lead_to_sp2[row_snp].append(other_snp)

                j -= 1

            # Scan right side of window
            j = idx + 1
            while j < len(df_chr_bp):
                other = df_chr_bp.loc[j]
                other_snp = other["SNP"]

                dist_bp = other["BP"] - row_bp
                if dist_bp > bp_window:
                    break

                if other_snp not in removed:
                    r2 = ld_r2(row_snp, other_snp)
                    if not np.isnan(r2) and r2 >= r2_threshold:
                        removed.add(other_snp)
                        lead_to_sp2[row_snp].append(other_snp)

                j += 1

    clumped = gwas_filtered[gwas_filtered["SNP"].isin(lead_snps)].copy()

    clumped["SP2"] = clumped["SNP"].map(
        lambda snp: ",".join(lead_to_sp2.get(snp, []))
    )
    
    clumped.to_csv(output_file_path, sep="\t", index=False)

In [3]:
clump('../plink_results/ps3_gwas.assoc.linear', '../python_results/gwas.clumped')

In [4]:
clump('../plink_results/ps3_gwas_covar.assoc.linear', '../python_results/gwas_covar.clumped')